# 04 · Ablaciones de chunking y contexto

Compara, sobre los reportes de benchmark, las decisiones de diseño del retrieval denso:

- **J1** (retrieval_text con contexto jurídico) vs **J2** (text crudo) vs **C1** (chunking clásico).
- **K_ONLY** vs **P_EXPAND_FULL** vs **P_EXPAND_BOUNDED**.
- Presupuestos **B4K / B8K / B12K**.
- Barrido de **k** (1, 3, 5, 10).

Consume `data/processed/reports/dense/benchmarks/<id>/`. No ejecuta retrieval aquí.

```bash
uv run python scripts/benchmark_dense_models.py --split development
```

In [ ]:
import json
from pathlib import Path

import pandas as pd

BENCH_ROOT = Path("data/processed/reports/dense/benchmarks")
runs = sorted(p for p in BENCH_ROOT.glob("*") if (p / "report.json").is_file())
if not runs:
    raise FileNotFoundError(
        f"No hay benchmarks en {BENCH_ROOT}. Ejecuta benchmark_dense_models.py (Gate C requerido)."
    )
run = runs[-1]
report = json.loads((run / "report.json").read_text(encoding="utf-8"))
print(f"benchmark_id: {report['benchmark_id']} | split: {report.get('split')}")
print(f"seed: {report.get('seed')}")

In [ ]:
metrics = pd.read_csv(run / "metrics.csv")
# Vistas J1/J2/C1 se distinguen por la columna 'view' (un bundle por view/modelo).
recall_cols = [c for c in metrics.columns if c.startswith("ParentRecall@")]
view_cols = [
    c
    for c in ["model_alias", "view", "ParentnDCG@10", "ParentHit@1", *recall_cols]
    if c in metrics.columns
]
metrics[view_cols].sort_values(["model_alias", "view"])  # noqa

## Lectura

La métrica primaria es **ParentnDCG@10** con controles ParentRecall@5, EvidenceRecall@5 y
ParentHit@1. Las diferencias entre vistas/estrategias deben acompañarse del IC 95 % por bootstrap
pareado (`report['bundles'][*]['primary_ci']`, semilla registrada) antes de afirmar que una opción
supera a otra. Guarda las figuras seleccionadas en `data/processed/reports/dense/figures/`.